In [1]:
import requests
import plotly.graph_objects as go
import plotly.io as pio

# open in browser
try:
    get_ipython()
    pio.renderers.default = "notebook_connected"
except NameError:
    pass

# median house prices in $'000 for June 2016, June 2021 and June 2026
prices = {
    "Jun 2016": {"Sydney": 895.0, "Melbourne": 610.0, "Brisbane": 495.0, "Adelaide": 440.0,
                 "Perth": 520.0, "Hobart": 365.0, "Darwin": 548.0, "Canberra": 623.0},
    "Jun 2021": {"Sydney": 1206.4, "Melbourne": 830.0, "Brisbane": 630.0, "Adelaide": 550.0,
                 "Perth": 540.0, "Hobart": 630.0, "Darwin": 565.0, "Canberra": 905.0},
    "Jun 2026": {"Sydney": 1487.6, "Melbourne": 850.0, "Brisbane": 1155.0, "Adelaide": 975.0,
                 "Perth": 1010.0, "Hobart": 750.0, "Darwin": 752.5, "Canberra": 1030.0},
}
periods = list(prices)

# get coastline data from GitHub repo
coast_URL = "https://raw.githubusercontent.com/johan/world.geo.json/master/countries/AUS.geo.json"
response = requests.get(coast_URL)
response.raise_for_status()
geojson = response.json()

# polygon and multipolygon geometries
geom = geojson["features"][0]["geometry"]
polygons = geom["coordinates"] if geom["type"] == "MultiPolygon" else [geom["coordinates"]]
coast_lon, coast_lat = [], []
for polygon in polygons:
    ring = polygon[0]
    coast_lon += [pt[0] for pt in ring] + [None]
    coast_lat += [pt[1] for pt in ring] + [None]

# city coordinates 
cities = {
    "Sydney":    (151.2093, -33.8688, "middle right"),
    "Melbourne": (144.9631, -37.8136, "bottom left"),
    "Brisbane":  (153.0251, -27.4698, "middle right"),
    "Adelaide":  (138.6007, -34.9285, "bottom left"),
    "Perth":     (115.8605, -31.9505, "middle right"),
    "Hobart":    (147.3272, -42.8821, "middle right"),
    "Darwin":    (130.8456, -12.4634, "middle right"),
    "Canberra":  (149.1300, -35.2809, "middle left"),
}
names = list(cities)
lon = [cities[c][0] for c in names]
lat = [cities[c][1] for c in names]
pos = [cities[c][2] for c in names]

# 6-class Reds from colorbrewer2.org
REDS_6 = ["#fee5d9", "#fcbba1", "#fc9272", "#fb6a4a", "#de2d26", "#a50f15"]
BREAKS = [500, 750, 1000, 1250, 1400]
LABELS = ["Under $500k", "$500k to 750k", "$750k to 1m",
          "$1m to 1.25m", "$1.25m to 1.4m", "Over $1.4m"]

def colour_for(price):
    return REDS_6[sum(price >= b for b in BREAKS)]

def city_trace(period):
    p = [prices[period][c] for c in names]
    return go.Scatter(
        x=lon, y=lat, mode="markers+text", showlegend=False,
        marker=dict(size=18, color=[colour_for(v) for v in p],
                    line=dict(width=1, color="#1f2937")),
        text=names,
        textposition=pos, textfont=dict(size=11, color="#111827"),
        customdata=list(zip(names, p)),
        hovertemplate="<b>%{customdata[0]}</b><br>Median house price: "
                      "$%{customdata[1]:,.0f}k<extra></extra>",
    )

coast = go.Scatter(x=coast_lon, y=coast_lat, mode="lines", fill="toself",
                   fillcolor="#e5e7eb", line=dict(color="#94a3b8", width=1),
                   hoverinfo="skip", showlegend=False)

# legend swatches
legend = [go.Scatter(x=[None], y=[None], mode="markers", name=n,
          marker=dict(size=12, color=c, line=dict(width=1, color="#1f2937")))
          for n, c in zip(LABELS[::-1], REDS_6[::-1])]

fig = go.Figure(
    data=[coast, city_trace(periods[-1])] + legend,
    frames=[go.Frame(data=[city_trace(p)], traces=[1], name=p) for p in periods],
)
fig.update_xaxes(visible=False, range=[112, 156])
fig.update_yaxes(visible=False, range=[-45.5, -9], scaleanchor="x", scaleratio=1.35)
fig.update_layout(
    title=dict(text="Median House Price by Australian Capital City", x=0.5),
    legend=dict(title="Median house price", x=0, xanchor="left", y=0.3),
    margin=dict(l=10, r=10, t=60, b=40), height=650,
    plot_bgcolor="white", paper_bgcolor="white",
    sliders=[dict(active=len(periods) - 1, currentvalue=dict(prefix="Period: ",
        font=dict(size=14, color="#111827")), font=dict(size=13, color="#111827"),
        pad=dict(t=10, l=120, r=120),
        steps=[dict(label=p, method="animate",
                    args=[[p], dict(mode="immediate", frame=dict(duration=0, redraw=True))])
               for p in periods])],
)

fig.show(auto_play=False)